In [1]:
import os
import sys

import math
import time
import datetime
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
# from YourDataset import YourDataset  # Import your custom dataset here
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
from torchinfo import summary
import torchprofile

from neuralop.models import TFNO3d

import pickle

torch.manual_seed(23)

scaler = GradScaler()

DTYPE = torch.float32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 200
plt.rcParams["font.family"] = "serif"

import scipy.stats as stats

/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
# Define your custom loss function here
class CustomLoss(nn.Module):
    def __init__(self):
        super(CustomLoss, self).__init__()

    def forward(self, y_pred, y_true):
        # Implement your custom loss calculation here
        # loss = torch.mean((y_pred - y_true) ** 2)  # Example: Mean Squared Error
        loss = torch.norm(y_true-y_pred, p=2)/torch.norm(y_true, p=2)
        return loss

class YourDataset(Dataset):
    def __init__(self, x, y, transform=None):
        self.x = x
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x_sample = self.x[idx]
        y_sample = self.y[idx]

        if self.transform:
            x_sample, y_sample = self.transform(x_sample, y_sample)

        return x_sample, y_sample
    
class Normalizer():
    def __init__(self, shift, scale):
        self.shift = torch.tensor(shift, dtype=torch.float32)
        self.scale = torch.tensor(scale, dtype=torch.float32)
    
    def normalize(self, x):
        return (x - self.shift)/self.scale
    
    def renormalize(self, x):
        return x*self.scale + self.shift

def get_xyt_grid(nx=None, ny=None, nt=None, bot=[0, 0, 0], top=[1, 1, 1], dtype='float32',
                 x_arr=None, y_arr=None, t_arr=None, dt=0.):
    '''
    Args:
        S: number of points on each spatial domain
        T: number of points on temporal domain including endpoint
        bot: list or tuple, lower bound on each dimension
        top: list or tuple, upper bound on each dimension

    Returns:
        (n_x, n_y, n_t, 3) array of grid points
    '''
    if x_arr is None:
        x_arr = np.linspace(bot[0], top[0], num=nx, endpoint=True)

    if y_arr is None:
        y_arr = np.linspace(bot[1], top[1], num=ny, endpoint=True)

    if t_arr is None:
        if dt is None:
            dt = (top[2] - bot[2]) / nt
        t_arr = np.linspace(bot[2] + dt, top[2], num=nt)

    x_grid, y_grid, t_grid = np.meshgrid(x_arr, y_arr, t_arr, indexing='ij')
    x_axis = np.ravel(x_grid)
    y_axis = np.ravel(y_grid)
    t_axis = np.ravel(t_grid)
    grid = np.stack([x_axis, y_axis, t_axis], axis=0).T

    x_grid, y_grid = np.meshgrid(x_arr, y_arr, indexing='ij')
    x_axis = np.ravel(x_grid)
    y_axis = np.ravel(y_grid)
    grid_space = np.stack([x_axis, y_axis], axis=0).T

    grids_dict = {'grid_x': x_arr, 
                  'grid_y': y_arr,
                  'grid_t': t_arr, 
                  'grid_space': grid_space, 
                  'grid': grid}
    for key in grids_dict.keys():
        grids_dict[key] = torch.tensor(grids_dict[key], dtype=eval('torch.' + dtype), device='cpu')
    return grids_dict



def _preprocess_fno(data_dict, Par):
    Grid_train = get_xyt_grid(Par['nx'], Par['ny'], Par['lf'], bot=[0, 0, 0], top=[1, 1, 1], dtype='float32')
    Grid_test  = get_xyt_grid(Par['nx'], Par['ny'], Par['lf'], bot=[0, 0, 0], top=[1, 1, 1], dtype='float32')
    
    data_dim = len(data_dict['x_train'].shape) - 2
    if data_dim != len(data_dict['x_test'].shape) - 2:
        raise ValueError('Data dimension mismatch between train and test sets.' +
                            f'Train: {data_dim}, Test: {len(data_dict["x_test"].shape)}')

    # Repeat shape is [1, time_steps_train, 1, 1, 1] for 3D data
    time_steps_train = Par['lf'] #self.configs.time_steps_train
    time_steps_val = Par['lf'] #self.configs.time_steps_inference if self.configs.scenario == 'hypersonics' \
    time_steps_test = Par['lf'] #self.configs.time_steps_inference
    # repeat_shape_train = [1, time_steps_train] + [1] * data_dim
    # repeat_shape_val = [1, time_steps_val] + [1] * data_dim
    # repeat_shape_test = [1, time_steps_test] + [1] * data_dim

    # data_dict['x_train'] = data_dict['x_train'].repeat(repeat_shape_train)
    # data_dict['x_val'] = data_dict['x_val'].repeat(repeat_shape_val)
    # data_dict['x_test'] = data_dict['x_test'].repeat(repeat_shape_test)

  

    for dataset in ['x_train', 'x_val', 'x_test']:
        n_samples = data_dict[dataset].shape[0]
        if True:
            grid = Grid_test if dataset == 'x_test' else Grid_train
            time_steps = time_steps_test if dataset == 'x_test' else time_steps_train

        grid_t = grid['grid_t'].reshape([1, time_steps] + [1] * data_dim)
        grid_t = grid_t.repeat(
            [n_samples, 1] + list(data_dict[dataset].shape[2:]))
        if data_dim == 1:
            x_shape = [1, 1] + list(data_dict[dataset].shape[2:3])
            grid_x = grid['grid_x'].reshape(
                x_shape).repeat(n_samples, time_steps, 1)
            data_dict[dataset] = torch.stack(
                [data_dict[dataset], grid_x, grid_t], dim=1)
        elif data_dim == 2:
            x_shape = [1, 1] + list(data_dict[dataset].shape[2:3]) + [1]
            y_shape = [1, 1, 1] + list(data_dict[dataset].shape[3:])

            grid_x = grid['grid_x'].reshape(x_shape).repeat(
                n_samples, time_steps, 1, y_shape[-1])
            grid_y = grid['grid_y'].reshape(y_shape).repeat(
                n_samples, time_steps, x_shape[-2], 1)

            data_dict[dataset] = torch.stack(
                [data_dict[dataset], grid_x, grid_y, grid_t], dim=1)
        elif data_dim == 3:
            x_shape = [1, 1] + list(data_dict[dataset].shape[2:3]) + [1, 1]
            y_shape = [1, 1, 1] + list(data_dict[dataset].shape[3:4]) + [1]
            z_shape = [1, 1, 1, 1] + list(data_dict[dataset].shape[4:])
            grid_x = grid['grid_x'].reshape(x_shape)
            grid_y = grid['grid_y'].reshape(y_shape)
            grid_z = grid['grid_z'].reshape(z_shape)

            grid_x = grid_x.repeat(
                n_samples, time_steps, 1, y_shape[-2], z_shape[-1])
            grid_y = grid_y.repeat(
                n_samples, time_steps, x_shape[-3], 1, z_shape[-1])
            grid_z = grid_z.repeat(
                n_samples, time_steps, x_shape[-3], y_shape[-2], 1)

            data_dict[dataset] = torch.stack(
                [data_dict[dataset], grid_x, grid_y, grid_z, grid_t], dim=1)
    return data_dict



def preprocess(traj, Par):
    x = sliding_window_view(traj[:,:-(Par['lf']),:,:], window_shape=Par['lb'], axis=1 ).transpose(0,1,4,2,3).reshape(-1,Par['lb'],Par['nx'], Par['ny'])
    y = sliding_window_view(traj[:,Par['lb']:,:,:], window_shape=Par['lf'], axis=1 ).transpose(0,1,4,2,3).reshape(-1,Par['lf'],Par['nx'], Par['ny'])
    t = np.linspace(0,1,Par['lf']).reshape(-1,1)

    nt = y.shape[1]
    n_samples = y.shape[0]

    # t = np.tile(t, [n_samples,1]).reshape(-1,)                                                     #[_*nt, ]
    # x = np.repeat(x,nt, axis=0)                                   #[_*nt, 1, 64, 64]
    # y = y.reshape(y.shape[0]*y.shape[1],1,y.shape[2],y.shape[3])  #[_*nt, 64, 64]


    print('x: ', x.shape)
    print('y: ', y.shape)
    print('t: ', t.shape)
    print()
    return x,y,t

In [3]:
res = 128
begin_time = time.time()
traj = np.load(f"/oscar/data/gk/voommen/no_diffusion/kolmogrov/data/alpha_1.5_tau_14_re_2007_N_1000_T_50_nt_200_nx_512/res_{res}/traj.npy") #[1000, nx, ny, nt]
traj = traj.transpose(0,3,1,2)
print(f"Data Loading Time: {time.time() - begin_time:.1f}s")


# each simulation, t \in [0s, 50s] in 201 timesteps

traj_train = traj[:800, :80][:, ::2]     #t \in [0s, 20s] in 40 steps
traj_val   = traj[800:900, :80][:, ::2]
traj_test  = traj[900:, :80][:, ::2]

Par = {}
# Par['nt'] = 100 
Par['nx'] = traj_train.shape[2]
Par['ny'] = traj_train.shape[3]
Par['nf'] = 1
Par['d_emb'] = 128

Par['lb'] = 20
Par['lf'] = 20 
# Par['temp'] = Par['nt'] - Par['lb'] - Par['lf'] + 2

Par['num_epochs'] = 500

begin_time = time.time()
print('\nTrain Dataset')
x_train, y_train, t_train = preprocess(traj_train, Par)
print('\nValidation Dataset')
x_val, y_val, t_val  = preprocess(traj_val, Par)
print('\nTest Dataset')
x_test, y_test, t_test  = preprocess(traj_test, Par)
print(f"Data Preprocess Time: {time.time() - begin_time:.1f}s")

t_min = np.min(t_train)
t_max = np.max(t_train)

Par['inp_shift'] = np.mean(x_train) 
Par['inp_scale'] = np.std(x_train)
Par['out_shift'] = np.mean(y_train)
Par['out_scale'] = np.std(y_train)
Par['t_shift']   = t_min
Par['t_scale']   = t_max - t_min

inp_normalizer = Normalizer(Par['inp_shift'], Par['inp_scale'])
out_normalizer = Normalizer(Par['out_shift'], Par['out_scale'])

with open('Par.pkl', 'wb') as f:
    pickle.dump(Par, f)

# sys.exit()
#########################

# Create custom datasets
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
t_train_tensor = torch.tensor(t_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

x_val_tensor   = torch.tensor(x_val,   dtype=torch.float32)
t_val_tensor   = torch.tensor(t_val,   dtype=torch.float32)
y_val_tensor   = torch.tensor(y_val,   dtype=torch.float32)

x_test_tensor  = torch.tensor(x_test,  dtype=torch.float32)
t_test_tensor  = torch.tensor(t_test,  dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32)

data_dict = {'x_train':x_train_tensor, 'x_val':x_val_tensor, 'x_test':x_test_tensor}
data_dict = _preprocess_fno(data_dict, Par)

x_train_tensor = torch.tensor(data_dict['x_train'].cpu().numpy(), dtype=torch.float32)
x_val_tensor   = torch.tensor(data_dict['x_val'].cpu().numpy()  , dtype=torch.float32)
x_test_tensor  = torch.tensor(data_dict['x_test'].cpu().numpy() , dtype=torch.float32)

print(f"x_train_tensor: {x_train_tensor.shape}")
print(f"x_val_tensor: {x_val_tensor.shape}")
print(f"x_test_tensor: {x_test_tensor.shape}")

train_dataset = YourDataset(x_train_tensor, y_train_tensor)
val_dataset = YourDataset(x_val_tensor, y_val_tensor)
test_dataset = YourDataset(x_test_tensor, y_test_tensor)

# Define data loaders
train_batch_size = 10
val_batch_size   = 10
test_batch_size  = 10
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=val_batch_size)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size)

Data Loading Time: 8.8s

Train Dataset
x:  (800, 20, 128, 128)
y:  (800, 20, 128, 128)
t:  (20, 1)


Validation Dataset
x:  (100, 20, 128, 128)
y:  (100, 20, 128, 128)
t:  (20, 1)


Test Dataset
x:  (100, 20, 128, 128)
y:  (100, 20, 128, 128)
t:  (20, 1)

Data Preprocess Time: 0.0s
x_train_tensor: torch.Size([800, 4, 20, 128, 128])
x_val_tensor: torch.Size([100, 4, 20, 128, 128])
x_test_tensor: torch.Size([100, 4, 20, 128, 128])


In [4]:
# Initialize your Unet2D model
model = TFNO3d(12, 12, 12, hidden_channels=20, in_channels=4, out_channels=1).cuda()


path_model = 'models/best_model.pt'
model.load_state_dict(torch.load(path_model))

print( summary(model, input_size=((1,)+x_train_tensor.shape[1:]) ) )

# Adjust the dimensions as per your model's input size
dummy_x = x_train_tensor[0:1]
dummy_input = dummy_x.to(device)

# Profile the model
flops = torchprofile.profile_macs(model, dummy_input)
print(f"FLOPs: {flops:.2e}")

# Define loss function and optimizer
criterion = CustomLoss()

Layer (type:depth-idx)                   Output Shape              Param #
TFNO3d                                   [1, 1, 20, 128, 128]      --
├─Lifting: 1-1                           [1, 20, 20, 128, 128]     --
│    └─Conv3d: 2-1                       [1, 20, 20, 128, 128]     100
├─FNOBlocks: 1-2                         [1, 20, 20, 128, 128]     1,200
│    └─ModuleList: 2-8                   --                        (recursive)
│    │    └─Conv3d: 3-1                  [1, 20, 20, 128, 128]     400
│    └─FactorizedSpectralConv: 2-3       [1, 20, 20, 128, 128]     2,793,936
├─FNOBlocks: 1-3                         [1, 20, 20, 128, 128]     (recursive)
│    └─ModuleList: 2-8                   --                        (recursive)
│    │    └─Conv3d: 3-2                  [1, 20, 20, 128, 128]     400
│    └─FactorizedSpectralConv: 2-5       [1, 20, 20, 128, 128]     (recursive)
├─FNOBlocks: 1-4                         [1, 20, 20, 128, 128]     (recursive)
│    └─ModuleList: 2-8     

/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::fft_rfftn". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::view_as_complex". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::tensordot". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::einsum". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::permute". Ski

# Speed Test

In [18]:
model.eval()
test_loss = 0.0
begin_time = time.time()
with torch.no_grad():
    for x, y_true in test_loader:
        x = x[0:1]
        y_true = y_true[0:1]
        with autocast():
            x1 = inp_normalizer.normalize(x)
            y_pred = out_normalizer.renormalize(model(x1.to(device))[:,0]) 
            loss   = criterion(y_pred, y_true.to(device))
        test_loss += loss.item()
        break

elapsed_time = time.time() - begin_time
print(f"elapsed time: {elapsed_time:.4f}s")

elapsed time: 0.0204s


# Sanity Check

In [10]:
y_true_ls = []
y_pred_ls = []

model.eval()
train_loss = 0.0
with torch.no_grad():
    for x, y_true in train_loader:
        with autocast():
            x = inp_normalizer.normalize(x)
            y_pred = out_normalizer.renormalize(model(x.to(device))[:,0]) 
            loss   = criterion(y_pred, y_true.to(device))
        train_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

train_loss /= len(train_loader)
print(f"Train Loss: {train_loss:.4e}")

TRAIN_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
TRAIN_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"TRAIN_TRUE: {TRAIN_TRUE.shape}, DTYPE: {TRAIN_TRUE.dtype}")
print(f"TRAIN_PRED: {TRAIN_PRED.shape}, DTYPE: {TRAIN_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
val_loss = 0.0
with torch.no_grad():
    for x, y_true in val_loader:
        with autocast():
            x = inp_normalizer.normalize(x)
            y_pred = out_normalizer.renormalize(model(x.to(device))[:,0]) 
            loss   = criterion(y_pred, y_true.to(device))
        val_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

val_loss /= len(val_loader)
print(f"Val Loss: {val_loss:.4e}")

VAL_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
VAL_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"VAL_TRUE: {VAL_TRUE.shape}, DTYPE: {VAL_TRUE.dtype}")
print(f"VAL_PRED: {VAL_PRED.shape}, DTYPE: {VAL_PRED.dtype}")


x_true_ls = []
y_true_ls = []
y_pred_ls = []

model.eval()
test_loss = 0.0
with torch.no_grad():
    for x, y_true in test_loader:
        with autocast():
            x1 = inp_normalizer.normalize(x)
            y_pred = out_normalizer.renormalize(model(x1.to(device))[:,0]) 
            loss   = criterion(y_pred, y_true.to(device))
        test_loss += loss.item()
        x_true_ls.append(x[:,0].detach().cpu().numpy())
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

test_loss /= len(test_loader)
print(f"Test Loss: {test_loss:.4e}")

TEST_X = np.concatenate(x_true_ls, axis=0).reshape(-1, Par['lb'], Par['nx'], Par['ny']).astype(np.float32)
TEST_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
TEST_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"TEST_X   : {TEST_X.shape}, DTYPE: {TEST_X.dtype}")
print(f"TEST_TRUE: {TEST_TRUE.shape}, DTYPE: {TEST_TRUE.dtype}")
print(f"TEST_PRED: {TEST_PRED.shape}, DTYPE: {TEST_PRED.dtype}")

Train Loss: 2.2655e-01
TRAIN_TRUE: (800, 20, 128, 128), DTYPE: float32
TRAIN_PRED: (800, 20, 128, 128), DTYPE: float32
Val Loss: 2.4554e-01
VAL_TRUE: (100, 20, 128, 128), DTYPE: float32
VAL_PRED: (100, 20, 128, 128), DTYPE: float32
Test Loss: 2.5267e-01
TEST_X   : (100, 20, 128, 128), DTYPE: float32
TEST_TRUE: (100, 20, 128, 128), DTYPE: float32
TEST_PRED: (100, 20, 128, 128), DTYPE: float32


In [11]:
np.save("TRAIN_TRUE.npy", TRAIN_TRUE)
np.save("TRAIN_PRED.npy", TRAIN_PRED)

np.save("VAL_TRUE.npy", VAL_TRUE)
np.save("VAL_PRED.npy", VAL_PRED)

np.save("TEST_X.npy", TEST_X)
np.save("TEST_TRUE.npy", TEST_TRUE)
np.save("TEST_PRED.npy", TEST_PRED)

In [6]:
sample1 = TRAIN_TRUE[1]
sample2 = TRAIN_TRUE[51]

err = np.abs(sample1 - sample2)
print(f"max err: {np.max(err)}")
print(f"min err: {np.min(err)}")

max err: 4.198629856109619
min err: 4.947185516357422e-06
